# Topic C — L3: Explicit Finite Difference Methods

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic 0 — Foundations](Topic-0_Foundations.ipynb) (§7 Black-Scholes PDE idea), [Topic B — Trees](Topic-B_L2_Trees_TimeDependentVol.ipynb) (trinomial probabilities — explicit FD is closely related).

**What this notebook covers**
1. Where the Black-Scholes PDE comes from
2. Why we use log-price $x = \log S$
3. How to replace derivatives by finite differences
4. The **explicit** scheme — formula, algorithm, link to trinomial trees
5. **Stability** and **error analysis** (consistency + stability $\Rightarrow$ convergence)
6. Central vs **upwind** differences (HW4)
7. Exam problems: Practice-Midterm P4(a), HW4 bond pricing, drift adjustment (Practice-Final P5)

**Exam map**

| Source | What it tests |
|---|---|
| L3.4 | Log-price PDE; $\nu = r - \sigma^2/2$ drift adjustment |
| L3.7–L3.10 | Explicit FD formula for $C_n^j$ |
| L3.16–L3.17 | Stability: $\Delta t \le (\Delta x)^2/\sigma^2$ |
| HW4 P1(b–f) | Explicit FD on Vasicek bond; upwind vs central |
| Practice-Midterm P4(a) | Black-Scholes PDE with general $r$ |
| Practice-Final P5 | GBM paths use $r - \sigma^2/2$, not $r$ |


In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

def bs_call(S, K, T, r, sigma):
    if T <= 0: return max(S - K, 0)
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / std
    d2 = d1 - std
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)


---

## 1. Why finite difference methods?

**Trees** (Topic B) approximate risk-neutral expectations on a discrete grid. **Finite difference (FD) methods** do something similar but start from the **PDE** that the option price satisfies.

| Method | Starting point | Best for |
|---|---|---|
| Trees | Discrete stock paths | Americans, barriers, local vol |
| Explicit FD | PDE in $(x,t)$ | Understanding stability/error; simple PDEs |
| Implicit / Crank-Nicolson (Topic D) | Same PDE | No harsh stability limit on $\Delta t$ |

**Plain English:** The option price $C(S,t)$ solves a differential equation. FD methods replace derivatives $\partial C/\partial t$, $\partial C/\partial S$, $\partial^2 C/\partial S^2$ by **algebraic formulas** involving nearby grid values. That turns the PDE into a recursive calculation on a grid — like a tree, but derived from calculus.


## 2. The Black-Scholes PDE — where it comes from

Under GBM with growth rate $R_{\mathrm{grow}}$ and discount rate $R_{\mathrm{disc}}$ (Topic 0 §5–6), the option price $\tilde C(S,t)$ satisfies:
$$
\frac{\partial \tilde C}{\partial t}
+ R_{\mathrm{disc}}\, S\,\frac{\partial \tilde C}{\partial S}
+ \tfrac12 \sigma^2 S^2\,\frac{\partial^2 \tilde C}{\partial S^2}
= R_{\mathrm{disc}}\,\tilde C.
$$

**Where does this come from?** (Sketch, not full proof.) Apply Itô's lemma to $\tilde C(S_t, t)$, then form a self-financing portfolio that is long the option and short $\Delta = \partial \tilde C/\partial S$ shares. The random terms cancel, and no-arbitrage forces the drift of the portfolio to equal the risk-free rate. That gives the PDE above.

**Terminal condition:** $\tilde C(S, T) = f(S)$ (the payoff at expiry).

For a **non-dividend** stock with $R_{\mathrm{grow}} = R_{\mathrm{disc}} = r$, this is the standard Black-Scholes PDE.


## 3. Log-price transform (L3.4) — why and how

### Why change variables?

The PDE in $S$ has coefficients that depend on $S$ (terms like $S\,\partial \tilde C/\partial S$ and $S^2\,\partial^2 \tilde C/\partial S^2$). That makes the grid awkward: stability constraints depend on $S$ (L3.18), forcing tiny time steps at high stock prices.

Define:
$$
x = \log S, \qquad C(x,t) = \tilde C(e^x, t).
$$

### The GBM in log space

If $dS_t = R_{\mathrm{grow}} S_t\,dt + \sigma S_t\,dW_t$, then Itô gives:
$$
d(\log S_t) = \underbrace{(R_{\mathrm{grow}} - \tfrac12\sigma^2)}_{\nu}\,dt + \sigma\,dW_t.
$$

We call $\nu$ the **drift of log-price**. It is **not** the same as $r$ or $R_{\mathrm{grow}}$ — the $-\tfrac12\sigma^2$ is the Jensen correction (Topic 0 §5).

### The PDE in log-price

After changing variables, the PDE becomes:
$$
\boxed{
\frac{\partial C}{\partial t}
+ \nu\,\frac{\partial C}{\partial x}
+ \tfrac12\sigma^2\,\frac{\partial^2 C}{\partial x^2}
= r\,C,
\qquad C(x,T) = f(e^x).
}
$$

**Key exam fact (Practice-Final P5, HW7):** When **simulating** GBM paths, use drift $\nu = r - \tfrac12\sigma^2$ in the exponent, **not** $r$. The code `S0*exp(r*dt + sigma*sqrt(dt)*Z)` is wrong; it should be `S0*exp((r - 0.5*sigma**2)*dt + sigma*sqrt(dt)*Z)`.


In [2]:
# Verify: wrong drift vs correct drift for lognormal mean
S0, r, sigma, T = 100, 0.03, 0.20, 1.0
n = 100000
Z = np.random.default_rng(0).normal(size=n)
S_wrong = S0 * np.exp(r*T + sigma*np.sqrt(T)*Z)
S_right = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
print(f"Theory E[S_T] = S0*exp(rT) = {S0*np.exp(r*T):.2f}")
print(f"Wrong drift sample mean:  {S_wrong.mean():.2f}")
print(f"Correct drift sample mean: {S_right.mean():.2f}")


Theory E[S_T] = S0*exp(rT) = 103.05
Wrong drift sample mean:  105.11
Correct drift sample mean: 103.03


## 4. The finite difference grid (L3.5)

We discretize time and log-price:

| Symbol | Meaning |
|---|---|
| $N$ | Number of time steps |
| $\Delta t = T/N$ | Time step size |
| $t_n = n\Delta t$ | Time level $n = 0,1,\ldots,N$ |
| $2J$ | Number of space steps |
| $\Delta x$ | Space step in log-price |
| $x_j = x_0 + j\Delta x$ | Log-price level, $j = -J,\ldots,J$ |
| $C_n^j$ | **Approximate** option value at $(x_j, t_n)$ |
| $C(x_j, t_n)$ | **True** PDE solution at that point |

**Direction of computation:** Start at expiry $n=N$ (terminal condition known). Step **backwards** in time to $n=0$.

**Boundary conditions:** At $j = \pm J$ (far from the strike), we impose values from the payoff or asymptotics. For a call: $C_{-J}^n \approx 0$, $C_J^n \approx e^{x_J} - K e^{-r(T-t_n)}$.


## 5. Replacing derivatives — Taylor derivation (HW4 P1d)

This is the algebra behind every FD scheme. Let $C(x,t)$ be smooth.

### First derivative — one-sided (forward)

$$
C(x+h, t) = C(x,t) + h\,C_x(x,t) + O(h^2).
$$
Rearrange:
$$
\frac{C(x+h,t) - C(x,t)}{h} = C_x(x,t) + O(h).
$$
**Accuracy:** $O(h)$ — first order.

### First derivative — central

$$
C(x+h,t) = C + h C_x + \tfrac{h^2}{2}C_{xx} + O(h^3),
$$
$$
C(x-h,t) = C - h C_x + \tfrac{h^2}{2}C_{xx} + O(h^3).
$$
Subtract and divide by $2h$:
$$
\boxed{\frac{C(x+h,t) - C(x-h,t)}{2h} = C_x(x,t) + O(h^2).}
$$
**Accuracy:** $O(h^2)$ — second order.

### Second derivative — central

Add the two expansions:
$$
\frac{C(x+h,t) - 2C(x,t) + C(x-h,t)}{h^2} = C_{xx}(x,t) + O(h^2).
$$

### Time derivative — forward (explicit scheme uses backward in time)

For explicit FD we evaluate spatial derivatives at the **newer** time level $t_{n+1}$ and solve for the **older** level $t_n$:
$$
\frac{\partial C}{\partial t}\bigg|_{t_n} \approx \frac{C_n^j - C_{n+1}^j}{\Delta t}.
$$
(Negative because we go backwards: $t_n < t_{n+1}$.)


## 6. The explicit FD scheme (L3.7–L3.10)

Plug finite differences into the log-price PDE at $(x_j, t_{n+1})$:

$$
\frac{C_n^j - C_{n+1}^j}{\Delta t}
+ \nu\,\frac{C_{n+1}^{j+1} - C_{n+1}^{j-1}}{2\Delta x}
+ \tfrac12\sigma^2\,\frac{C_{n+1}^{j+1} - 2C_{n+1}^j + C_{n+1}^{j-1}}{(\Delta x)^2}
= r\,C_n^j.
$$

(Prof. Lee replaces $r C_{n+1}^j$ by $r C_n^j$ on the right — same convergence rate.)

### Solve for $C_n^j$ (the unknown older values)

Rearranging:
$$
\boxed{
C_n^j = \frac{1}{1 + r\Delta t}\Bigl(q_u\,C_{n+1}^{j+1} + q_m\,C_{n+1}^j + q_d\,C_{n+1}^{j-1}\Bigr)
}
$$
where
$$
q_u = \tfrac12\Bigl[\tfrac{\sigma^2\Delta t}{(\Delta x)^2} + \tfrac{\nu\Delta t}{\Delta x}\Bigr],\quad
q_m = 1 - \tfrac{\sigma^2\Delta t}{(\Delta x)^2},\quad
q_d = \tfrac12\Bigl[\tfrac{\sigma^2\Delta t}{(\Delta x)^2} - \tfrac{\nu\Delta t}{\Delta x}\Bigr].
$$

### Algorithm (memorize this procedure)

1. **Initialize** at $n=N$: $C_N^j = f(e^{x_j})$ (payoff).
2. **For** $n = N-1, N-2, \ldots, 0$:
   - For each interior $j$, compute $C_n^j$ from the formula above using column $n+1$.
   - Apply boundary conditions at $j = \pm J$.
3. **Answer:** $C_0^0$ at the node closest to $(\log S_0, 0)$.

### Link to trinomial trees (L3.10)

This formula is a **trinomial tree in disguise**: weights $q_u, q_m, q_d$ play the role of $p_u, p_m, p_d$, and $1/(1+r\Delta t) \approx e^{-r\Delta t}$. Topic B's tree and Topic C's explicit FD are the same idea from two angles.


## 7. Stability — when explicit FD blows up (L3.16–L3.17)

For the weights to be **nonnegative** (probabilities cannot be negative):

$$
q_u, q_d \ge 0 \quad\Rightarrow\quad \Delta x \le \frac{\sigma^2}{|\nu|},
$$
$$
q_m \ge 0 \quad\Rightarrow\quad \boxed{\Delta t \le \frac{(\Delta x)^2}{\sigma^2}}.
$$

**Intuition (trinomial picture):** Each step tries to produce variance $\sigma^2\Delta t$ using jumps of size $\Delta x$. If $\sigma^2\Delta t > (\Delta x)^2$, you cannot fit that much variance into too-small jumps — negative weights appear and errors **explode**.

**Practical recommendation:** $\Delta x = \sigma\sqrt{3\Delta t}$ (same as trinomial tree). Then $\Delta t = (\Delta x)^2/(3\sigma^2) < (\Delta x)^2/\sigma^2$ — safely inside the stability region.

**Why implicit methods matter (Topic D preview):** Explicit FD requires $\Delta t = O((\Delta x)^2)$ — very small time steps. Implicit and Crank-Nicolson schemes remove this restriction.


In [3]:
def explicit_fd_call(S0, K, T, r, sigma, N=400, J=400):
    nu = r - 0.5*sigma**2
    dt = T / N
    dx = sigma * np.sqrt(3*dt)
    x0 = np.log(S0)
    xs = x0 + np.arange(-J, J+1) * dx  # index J is at log(S0)
    C = np.maximum(np.exp(xs) - K, 0.0)
    qu = 0.5*(sigma**2*dt/dx**2 + nu*dt/dx)
    qm = 1 - sigma**2*dt/dx**2
    qd = 0.5*(sigma**2*dt/dx**2 - nu*dt/dx)
    disc = 1 / (1 + r*dt)
    for n in range(N-1, -1, -1):
        C_new = np.empty_like(C)
        C_new[1:-1] = disc*(qu*C[2:] + qm*C[1:-1] + qd*C[:-2])
        C_new[0] = 0.0
        C_new[-1] = np.exp(xs[-1]) - K*np.exp(-r*(T - n*dt))
        C = C_new
    return C[J]

S0, K, T, r, sigma = 100, 100, 0.5, 0.05, 0.30
fd_price = explicit_fd_call(S0, K, T, r, sigma)
bs_price = bs_call(S0, K, T, r, sigma)
print(f"Explicit FD call price: {fd_price:.4f}")
print(f"Black-Scholes price:    {bs_price:.4f}")


Explicit FD call price: 9.6297
Black-Scholes price:    9.6349


## 8. Error analysis — the exam trilogy (L3.12–L3.15, L3.29)

Three concepts — know the definitions and the **Lax equivalence** slogan.

### (1) Local truncation error (consistency)

Plug the **true** PDE solution into the FD equation. The mismatch is $L(x,t)$. If $L \to 0$ as $\Delta t, \Delta x \to 0$, the scheme is **consistent**.

For explicit FD: $L = O(\Delta t) + O((\Delta x)^2)$ — first order in time, second in space.

### (2) Stability

Errors at one time level must not **amplify** as we step backwards. For explicit FD with $q_u, q_m, q_d \ge 0$:
$$
E_n \le E_{n+1} + \Delta t\,\bar L,
$$
where $E_n = \max_j |C_n^j - C(x_j,t_n)|$. Errors accumulate but stay bounded if $\bar L$ is small.

### (3) Convergence

$\max_n E_n \to 0$. **Lax Equivalence Theorem:** consistency + stability $\Rightarrow$ convergence.

**Convergence rate:** $O(\Delta t) + O((\Delta x)^2)$ for explicit FD.

**Caveats (L3.21):** Discontinuous payoffs (digitals) break smoothness; boundary errors; non-constant $\sigma$ needs extra care.


## 9. Central vs upwind differences (HW4)

HW4 prices a **Vasicek bond** with PDE (interest rate $r$ as state variable):
$$
\frac{\partial C}{\partial t} + \kappa(\theta - r)\frac{\partial C}{\partial r} + \tfrac12\sigma^2\frac{\partial^2 C}{\partial r^2} = r\,C.
$$

The drift $\kappa(\theta - r)$ can be **strong** and **signed**.

| Scheme | $\partial C/\partial r$ approximated by | Accuracy | Stability when drift is strong |
|---|---|---|---|
| **Central** | $(C_{j+1} - C_{j-1})/(2\Delta r)$ | $O(\Delta r^2)$ | Can be unstable — needs very small $\Delta t$ |
| **Upwind** | One-sided, in direction of drift | $O(\Delta r)$ | More stable — less restrictive on grid |

**Upwind rule:** If drift $\kappa(\theta - r_j) \ge 0$, use backward difference $(C_{j+1}-C_j)/\Delta r$. If drift $< 0$, use forward difference $(C_j - C_{j-1})/\Delta r$.

**HW4 P1(e) result:** Central difference gives bond price $\approx 0.7661$ (correct). Upwind with same grid is **very inaccurate** — lower order but better stability in theory; here central wins on accuracy when stable.

**HW4 P1(f) fill-in-the-blanks:**
- Upwind is **less** accurate than central (considering truncation error only).
- But upwind allows **greater** freedom in choosing grid spacing when drift is strong.


## 10. Practice-Midterm P4(a) — PDE with general $r$

**Problem:** For GBM with volatility $\sigma$, find the PDE for contract price $C(S,t)$ when $r$ may be nonzero.

**Starting point:** The model PDE in the problem (with $r=0$) is:
$$
\frac{\partial C}{\partial t} - \tfrac12 S^2\sigma^2\,\frac{\partial^2 C}{\partial S^2} = 0.
$$

**With general $r$** (from L3.4 / HW5 / L4.11), add drift and discount terms:
$$
\boxed{
\frac{\partial C}{\partial t}
+ rS\,\frac{\partial C}{\partial S}
+ \tfrac12\sigma^2 S^2\,\frac{\partial^2 C}{\partial S^2}
= r\,C.
}
$$

Each term has meaning:
- $\partial C/\partial t$: time decay (theta)
- $rS\,\partial C/\partial S$: drift of the stock under $\mathbb Q$
- $\tfrac12\sigma^2 S^2\,\partial^2 C/\partial S^2$: diffusion (gamma term)
- $rC$: discounting at risk-free rate


## 11. Worked example — Practice-Midterm P4(b) preview

Practice-Midterm P4(b) uses **Crank-Nicolson** (Topic D), but the idea is the same grid. Given partial Crank-Nicolson output at $S=148,150,152$ and $t=1.1,1.2$, reconstruct hidden $C(150, 1.1)$.

For **explicit** FD, the pattern is identical: each interior node satisfies a linear relation between neighbors. If you know the scheme and surrounding values, you can solve one equation for the unknown.

*(Full Crank-Nicolson reconstruction is in Topic D.)*


## 12. Exam-style exercises

**E1.** For $r=0.05$, $\sigma=0.3$, $T=0.5$, $N=100$, what is $\Delta t$? If $\Delta x = \sigma\sqrt{3\Delta t}$, verify $\Delta t \le (\Delta x)^2/\sigma^2$.

**E2.** Write $q_u + q_m + q_d$ for the explicit scheme. What should it equal (approximately)?

**E3.** Why does explicit FD in $S$ (not $\log S$) waste resources at high $S$?

**E4.** Practice-Final P5: the line `paths = S0*np.exp(r*dt*...)` is wrong. What should the drift term be, and why?


### Solutions

<details><summary><b>E1</b></summary>

$\Delta t = 0.5/100 = 0.005$. $\Delta x = 0.3\sqrt{3\cdot 0.005} = 0.03674$. $(\Delta x)^2/\sigma^2 = 0.015$. Since $\Delta t = 0.005 < 0.015$, stable.
</details>

<details><summary><b>E2</b></summary>

$q_u + q_m + q_d = 1$. Then $C_n^j = \frac{1}{1+r\Delta t}(q_u C_{n+1}^{j+1} + q_m C_{n+1}^j + q_d C_{n+1}^{j-1})$ — weights sum to 1 before discounting.
</details>

<details><summary><b>E3</b></summary>

Stability requires $\Delta t \le (\Delta S)^2/(\sigma^2 S^2)$. At large $S$, this forces $\Delta t$ very small for the whole grid (L3.18).
</details>

<details><summary><b>E4</b></summary>

Use $(r - 0.5\sigma^2)$ instead of $r$ in the log-drift. GBM log increments have drift $\nu = r - \sigma^2/2$, not $r$ (L3.4).
</details>


## 13. Cheat sheet — Topic C

| # | Formula / fact | Use |
|---|---|---|
| 1 | Log-price PDE: $C_t + \nu C_x + \tfrac12\sigma^2 C_{xx} = rC$ | Starting point |
| 2 | $\nu = r - \tfrac12\sigma^2$ | Drift adjustment; GBM simulation |
| 3 | $C_n^j = \frac{1}{1+r\Delta t}(q_u C_{n+1}^{j+1} + q_m C_{n+1}^j + q_d C_{n+1}^{j-1})$ | Explicit scheme |
| 4 | $\Delta t \le (\Delta x)^2/\sigma^2$ | Stability |
| 5 | $\Delta x = \sigma\sqrt{3\Delta t}$ | Recommended grid |
| 6 | Central: $O(h^2)$; one-sided: $O(h)$ | HW4 truncation error |
| 7 | Consistency + stability $\Rightarrow$ convergence | Lax theorem |
| 8 | Rate: $O(\Delta t) + O((\Delta x)^2)$ | Explicit FD error |

---

*Next:* **Topic D — L4: Implicit & Crank-Nicolson FD** (Practice-Midterm P4b–c).
